# Advanced Natural Language Interface for Child Welfare Data

## Purpose
Enable welfare organization managers, directors, and social workers to explore child protection data using natural language queries without technical expertise.

## Key Features
1. **Multi-Dataset Integration**: Queries across time series, features, and allocations
2. **Intelligent Query Understanding**: Handles variations in phrasing, typos, and context
3. **Actionable Responses**: Provides insights ready for decision-making
4. **Contextual Follow-ups**: Maintains conversation context
5. **Visual Summaries**: Shows data tables and recommendations

## Target Users
- Welfare organization chairmen and directors
- District managers and coordinators
- Social workers and field officers
- Policy makers and donors

## Sample Questions Supported
- "Which district needs the most resources?"
- "Show me districts where abuse is increasing"
- "What's the allocation for Hambantota?"
- "Compare Matara and Colombo on infrastructure"
- "Which districts are high priority?"
- "How has abuse changed in the last 5 years?"

## 1. Setup and Dependencies

In [ ]:
# Install required libraries

import sys

try:
    from sentence_transformers import SentenceTransformer
    print("sentence-transformers already installed")
except ImportError:
    print("Installing sentence-transformers...")
    !pip install -q sentence-transformers
    print("Installation complete")

try:
    import fuzzywuzzy
    print("fuzzywuzzy already installed")
except ImportError:
    print("Installing fuzzywuzzy...")
    !pip install -q fuzzywuzzy python-Levenshtein
    print("Installation complete")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

from sentence_transformers import SentenceTransformer, util
from fuzzywuzzy import fuzz, process
import re
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 2. Load All Datasets

Loading data from all phases to enable comprehensive queries.

In [ ]:
# Configuration - Update these paths to match your file locations
DATA_PATHS = {
    'time_series': '/content/drive/My Drive/output/combined_districts.csv',
    'features': '/content/drive/My Drive/output/district_features.csv',
    'allocations': '/content/drive/My Drive/output/allocation_recommendations_2025.csv',
    'need_scores': '/content/drive/My Drive/output/district_need_scores.csv'
}

# Default budget for allocation calculations (50 million rupees)
DEFAULT_BUDGET = 50_000_000

print("Configuration set")
print(f"Default budget: {DEFAULT_BUDGET:,} rupees")

In [ ]:
# Load all datasets
# WHY: Different queries need different data sources
#      - Time series: For trend questions
#      - Features: For comparative analysis
#      - Allocations: For resource allocation queries

datasets = {}

print("Loading datasets...\n")

for key, path in DATA_PATHS.items():
    try:
        datasets[key] = pd.read_csv(path)
        print(f"Loaded {key}: {datasets[key].shape}")
    except FileNotFoundError:
        print(f"WARNING: {key} not found at {path}")
        datasets[key] = None
    except Exception as e:
        print(f"ERROR loading {key}: {str(e)}")
        datasets[key] = None

# Get list of available districts
# WHY: Need to recognize district names in queries
if datasets['features'] is not None:
    DISTRICTS = sorted(datasets['features']['District'].unique().tolist())
    print(f"\nDistricts available: {DISTRICTS}")
else:
    DISTRICTS = []
    print("\nWARNING: No district data loaded")

print("\nDataset loading complete")

## 3. Initialize NLP Model

Using SentenceTransformer for semantic understanding of queries.

In [ ]:
# Load semantic similarity model
# WHY: all-MiniLM-L6-v2 is lightweight but effective for semantic matching
#      - Fast inference (important for interactive use)
#      - Good generalization to new phrases
#      - Works well with short queries

print("Loading NLP model...\n")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("NLP model loaded successfully")
print("Model: all-MiniLM-L6-v2")

## 4. Query Intent Classification System

Defines query templates organized by manager intent and use case.

In [ ]:
# Define query intent templates
# WHY: Organizes queries by what stakeholders actually want to know
#      Each template maps to a specific analysis function
#
# STRUCTURE:
# - intent: What the user wants to know
# - examples: Multiple ways to phrase the same question
# - handler: Function that generates the answer

QUERY_INTENTS = {
    'allocation_by_district': {
        'examples': [
            "What is the allocation for {district}?",
            "How much budget does {district} get?",
            "Show me {district} allocation",
            "Resources allocated to {district}",
            "What percentage goes to {district}?"
        ],
        'requires_district': True,
        'handler': 'get_district_allocation'
    },

    'highest_priority': {
        'examples': [
            "Which district needs the most resources?",
            "Show me the highest priority district",
            "Which area is most critical?",
            "Where should we allocate most?",
            "Most urgent district",
            "Top priority for funding"
        ],
        'requires_district': False,
        'handler': 'get_highest_priority'
    },

    'all_allocations': {
        'examples': [
            "Show all district allocations",
            "Complete allocation breakdown",
            "How is the budget distributed?",
            "List all district allocations",
            "Full allocation table"
        ],
        'requires_district': False,
        'handler': 'get_all_allocations'
    },

    'increasing_abuse': {
        'examples': [
            "Which districts have increasing abuse cases?",
            "Show me districts where abuse is getting worse",
            "Districts with worsening trends",
            "Where is abuse rising?",
            "Which areas show upward trends?"
        ],
        'requires_district': False,
        'handler': 'get_increasing_abuse'
    },

    'decreasing_abuse': {
        'examples': [
            "Which districts are improving?",
            "Show me where abuse is decreasing",
            "Districts with positive trends",
            "Where is the situation getting better?",
            "Success stories"
        ],
        'requires_district': False,
        'handler': 'get_decreasing_abuse'
    },

    'compare_districts': {
        'examples': [
            "Compare {district1} and {district2}",
            "How do {district1} and {district2} differ?",
            "Show me differences between {district1} and {district2}",
            "{district1} versus {district2}",
            "Comparison of {district1} and {district2}"
        ],
        'requires_district': True,
        'handler': 'compare_districts'
    },

    'infrastructure_poor': {
        'examples': [
            "Which districts have poor infrastructure?",
            "Show me districts with infrastructure problems",
            "Where is infrastructure worst?",
            "Districts needing infrastructure investment",
            "Poor student-teacher ratios"
        ],
        'requires_district': False,
        'handler': 'get_poor_infrastructure'
    },

    'high_current_abuse': {
        'examples': [
            "Which districts have the most abuse cases now?",
            "Highest current abuse rates",
            "Show me worst performing districts",
            "Most abuse cases currently",
            "Districts with highest abuse"
        ],
        'requires_district': False,
        'handler': 'get_high_current_abuse'
    },

    'tier_classification': {
        'examples': [
            "Show me districts by priority tier",
            "Which districts are critical?",
            "List tier 1 districts",
            "Show tier classifications",
            "Critical vs high vs moderate districts"
        ],
        'requires_district': False,
        'handler': 'get_tier_classification'
    },

    'district_profile': {
        'examples': [
            "Tell me about {district}",
            "Show me {district} details",
            "Complete profile for {district}",
            "What's happening in {district}?",
            "{district} overview"
        ],
        'requires_district': True,
        'handler': 'get_district_profile'
    },

    'trend_analysis': {
        'examples': [
            "Show me abuse trends over time",
            "How has abuse changed in the last 5 years?",
            "Trends in {district}",
            "Historical patterns",
            "Changes from 2012 to 2024"
        ],
        'requires_district': False,
        'handler': 'get_trend_analysis'
    },

    'budget_scenario': {
        'examples': [
            "What if budget is 75 million?",
            "Calculate allocation for 60 million budget",
            "Show me allocation with different budget",
            "Budget scenario analysis",
            "How would allocation change with more funding?"
        ],
        'requires_district': False,
        'handler': 'calculate_budget_scenario'
    }
}

print(f"Query intent system initialized with {len(QUERY_INTENTS)} intent categories")
print("\nSupported query types:")
for intent_name in QUERY_INTENTS.keys():
    print(f"  - {intent_name.replace('_', ' ').title()}")

## 5. District Name Recognition

Extracts district names from queries using fuzzy matching to handle typos.

In [ ]:
def extract_district_names(query: str, districts: List[str]) -> List[str]:
    """
    Extract district names from user query using fuzzy matching.

    WHY FUZZY MATCHING:
    - Users may misspell district names (Hambantota vs Hambantotta)
    - Case-insensitive matching needed
    - Handles partial matches

    Args:
        query: User's natural language query
        districts: List of valid district names

    Returns:
        List of matched district names
    """
    if not districts:
        return []

    matched_districts = []
    query_lower = query.lower()

    for district in districts:
        district_lower = district.lower()

        # Exact match (case-insensitive)
        if district_lower in query_lower:
            matched_districts.append(district)
            continue

        # Fuzzy match with threshold
        # WHY 80: Allows for small typos while avoiding false positives
        similarity = fuzz.partial_ratio(district_lower, query_lower)
        if similarity >= 80:
            matched_districts.append(district)

    return matched_districts


print("District name recognition functions defined")

Extracts budget amount from queries using fuzzy matching to handle typos.

In [ ]:
def extract_budget_amount(query: str) -> Optional[float]:
    """
    Extract budget amount from query.

    WHY: For budget scenario queries like "What if budget is 75 million?"

    Handles:
    - "75 million" -> 75000000
    - "75M" -> 75000000
    - "7.5 crore" -> 75000000
    - "75000000" -> 75000000

    Args:
        query: User query

    Returns:
        Budget amount in rupees or None if not found
    """
    # Pattern for millions
    million_pattern = r'(\d+\.?\d*)\s*(?:million|m)'
    match = re.search(million_pattern, query.lower())
    if match:
        return float(match.group(1)) * 1_000_000

    # Pattern for crores
    crore_pattern = r'(\d+\.?\d*)\s*(?:crore|cr)'
    match = re.search(crore_pattern, query.lower())
    if match:
        return float(match.group(1)) * 10_000_000

    # Pattern for lakhs
    lakh_pattern = r'(\d+\.?\d*)\s*(?:lakh|lac)'
    match = re.search(lakh_pattern, query.lower())
    if match:
        return float(match.group(1)) * 100_000

    # Pattern for raw numbers (must be large enough to be a budget)
    number_pattern = r'(\d{7,})'
    match = re.search(number_pattern, query)
    if match:
        return float(match.group(1))

    return None

## 6. Query Intent Classifier

Matches user query to the most appropriate intent using semantic similarity.

In [ ]:
def classify_intent(query: str, debug: bool = False) -> Tuple[str, float, Dict]:
    """
    Classify user query into one of the predefined intents.

    WHY SEMANTIC SIMILARITY:
    - Handles paraphrasing ("Show allocation" vs "What's the budget for")
    - Works with new phrasings not in templates
    - More flexible than keyword matching

    PROCESS:
    1. Embed user query
    2. Embed all template examples
    3. Find most similar template
    4. Return corresponding intent

    Args:
        query: User's natural language query
        debug: If True, print similarity scores

    Returns:
        (intent_name, confidence_score, metadata)
    """
    # Encode user query
    query_embedding = model.encode(query, convert_to_tensor=True)

    best_match = None
    best_score = 0
    best_intent = None

    # Compare against all intent examples
    for intent_name, intent_data in QUERY_INTENTS.items():
        # Encode all examples for this intent
        examples = intent_data['examples']
        example_embeddings = model.encode(examples, convert_to_tensor=True)

        # Calculate similarities
        similarities = util.cos_sim(query_embedding, example_embeddings)[0]

        # Get best match for this intent
        max_sim = float(similarities.max())
        max_idx = int(similarities.argmax())

        if debug:
            print(f"{intent_name}: {max_sim:.4f} (matched: '{examples[max_idx]}')")

        # Update best overall match
        if max_sim > best_score:
            best_score = max_sim
            best_intent = intent_name
            best_match = examples[max_idx]

    metadata = {
        'matched_template': best_match,
        'requires_district': QUERY_INTENTS[best_intent]['requires_district'],
        'handler': QUERY_INTENTS[best_intent]['handler']
    }

    return best_intent, best_score, metadata


print("Intent classification function defined")

## 7. Response Generation Functions

Each function handles a specific type of query and generates appropriate response.

In [ ]:
def format_rupees(amount: float) -> str:
    """
    Format rupee amounts in Sri Lankan convention.

    WHY: Local convention for readability
    - 1 lakh = 100,000
    - 1 crore = 10,000,000
    """
    if amount >= 10_000_000:
        return f"{amount/10_000_000:.2f} Crore"
    elif amount >= 100_000:
        return f"{amount/100_000:.2f} Lakh"
    else:
        return f"{amount:,.0f}"

In [ ]:
def get_district_allocation(districts: List[str], budget: float = DEFAULT_BUDGET) -> str:
    """
    Get allocation details for specific district(s).

    WHY: Most common manager query - "How much does X get?"
    """
    if not districts:
        return "Please specify which district you want to know about."

    if datasets['allocations'] is None:
        return "Allocation data not available. Please run Phase 4 first."

    response_parts = []

    for district in districts:
        district_data = datasets['allocations'][
            datasets['allocations']['District'].str.lower() == district.lower()
        ]

        if len(district_data) == 0:
            response_parts.append(f"No data found for {district}")
            continue

        row = district_data.iloc[0]

        # Calculate budget amount based on percentage
        allocation_pct = row.get('Allocation_Percent', 0)
        budget_amount = (allocation_pct / 100) * budget

        response = f"""
ALLOCATION FOR {district.upper()}
{'-'*50}
Priority Rank: {row.get('Rank', 'N/A')}
Need Score: {row.get('Need_Score', 'N/A'):.1f}/100
Priority Tier: {row.get('Priority_Tier', 'N/A')}
Allocation: {allocation_pct:.1f}%
Budget Amount: {format_rupees(budget_amount)} Rupees

INTERPRETATION:
{interpret_allocation(row, district)}
"""
        response_parts.append(response)

    return "\n".join(response_parts)

In [ ]:
def interpret_allocation(row: pd.Series, district: str) -> str:
    """
    Provide human-readable interpretation of allocation.

    WHY: Managers need actionable insights, not just numbers
    """
    tier = row.get('Priority_Tier', '')
    need_score = row.get('Need_Score', 0)

    if 'Critical' in tier:
        return f"{district} is classified as CRITICAL PRIORITY. This district requires immediate intervention with enhanced monitoring, infrastructure investment, and emergency response programs."
    elif 'High' in tier:
        return f"{district} is classified as HIGH PRIORITY. Recommended actions include preventive programs, capacity building, and community engagement initiatives."
    else:
        return f"{district} is classified as MODERATE PRIORITY. Continue standard programs with routine monitoring and infrastructure maintenance."

In [ ]:
def get_highest_priority() -> str:
    """
    Identify district with highest need.

    WHY: Quick answer to "Where should we focus first?"
    """
    if datasets['allocations'] is None:
        return "Allocation data not available."

    df = datasets['allocations'].copy()
    top_district = df[df['Rank'] == 1].iloc[0]

    district = top_district['District']
    need_score = top_district['Need_Score']
    allocation_pct = top_district['Allocation_Percent']
    tier = top_district['Priority_Tier']

    return f"""
HIGHEST PRIORITY DISTRICT: {district.upper()}
{'-'*50}

Need Score: {need_score:.1f}/100 (Highest among all districts)
Priority Tier: {tier}
Recommended Allocation: {allocation_pct:.1f}%

WHY THIS DISTRICT IS TOP PRIORITY:
{get_priority_explanation(district)}

RECOMMENDED IMMEDIATE ACTIONS:
- Deploy emergency response teams
- Conduct comprehensive needs assessment
- Invest in infrastructure improvements
- Launch intensive community awareness programs
- Establish enhanced monitoring systems
"""

In [ ]:
def get_priority_explanation(district: str) -> str:
    """
    Explain why a district is high priority.

    WHY: Justifies allocation decisions with data
    """
    if datasets['features'] is None:
        return "Based on composite need score analysis."

    district_features = datasets['features'][
        datasets['features']['District'].str.lower() == district.lower()
    ]

    if len(district_features) == 0:
        return "Based on comprehensive need assessment."

    row = district_features.iloc[0]
    reasons = []

    # Check current severity
    if 'abuse_rate_per_1000' in row:
        rate = row['abuse_rate_per_1000']
        if rate > datasets['features']['abuse_rate_per_1000'].quantile(0.75):
            reasons.append(f"High current abuse rate ({rate:.2f} per 1000 children)")

    # Check trends
    if 'abuse_trend_slope' in row:
        slope = row['abuse_trend_slope']
        if slope > 0:
            reasons.append(f"Worsening trend (abuse increasing by {slope:.1f} cases/year)")

    # Check infrastructure
    if 'student_teacher_ratio' in row:
        ratio = row['student_teacher_ratio']
        if ratio > datasets['features']['student_teacher_ratio'].quantile(0.75):
            reasons.append(f"Poor infrastructure (student-teacher ratio: {ratio:.1f}:1)")

    if reasons:
        return "\n".join([f"- {r}" for r in reasons])
    else:
        return "Based on comprehensive multi-factor analysis of need indicators."

In [ ]:
def get_all_allocations(budget: float = DEFAULT_BUDGET) -> str:
    """
    Show complete allocation table for all districts.

    WHY: Overview for budget planning meetings
    """
    if datasets['allocations'] is None:
        return "Allocation data not available."

    df = datasets['allocations'].copy()
    df = df.sort_values('Rank')

    # Calculate budget amounts
    df['Budget_Amount'] = (df['Allocation_Percent'] / 100) * budget
    df['Budget_Formatted'] = df['Budget_Amount'].apply(format_rupees)

    # Create display table
    display_df = df[[
        'Rank', 'District', 'Priority_Tier',
        'Need_Score', 'Allocation_Percent', 'Budget_Formatted'
    ]].copy()

    display_df.columns = [
        'Rank', 'District', 'Tier',
        'Need Score', 'Allocation %', 'Budget Amount'
    ]

    table = display_df.to_string(index=False)

    return f"""
2025 RESOURCE ALLOCATION SUMMARY
Total Budget: {format_rupees(budget)} Rupees
{'='*80}

{table}

{'='*80}
Total: 100.0% | {format_rupees(budget)} Rupees
"""


print("Basic response functions defined")

In [ ]:
# More response functions

def get_increasing_abuse() -> str:
    """
    Show districts where abuse is increasing.

    WHY: Identifies emerging problems needing preventive action
    """
    if datasets['features'] is None:
        return "Feature data not available."

    df = datasets['features'].copy()

    if 'abuse_trend_slope' not in df.columns:
        return "Trend data not available."

    # Filter districts with positive slope (increasing)
    increasing = df[df['abuse_trend_slope'] > 0].copy()

    if len(increasing) == 0:
        return "Good news! No districts show increasing abuse trends. All districts are stable or improving."

    # Sort by severity of increase
    increasing = increasing.sort_values('abuse_trend_slope', ascending=False)

    response = "DISTRICTS WITH INCREASING ABUSE TRENDS\n"
    response += "="*60 + "\n\n"

    for idx, row in increasing.iterrows():
        district = row['District']
        slope = row['abuse_trend_slope']

        current_rate = row.get('abuse_rate_per_1000', 0)
        pct_change_3yr = row.get('abuse_pct_change_3yr', 0)

        response += f"""
{district.upper()}
  Trend: +{slope:.1f} cases/year (INCREASING)
  Current Rate: {current_rate:.2f} per 1000 children
  3-Year Change: {pct_change_3yr:+.1f}%
  RECOMMENDATION: Urgent preventive intervention needed

"""

    return response

In [ ]:
def get_decreasing_abuse() -> str:
    """
    Show districts where abuse is decreasing.

    WHY: Identifies successful interventions to replicate
    """
    if datasets['features'] is None:
        return "Feature data not available."

    df = datasets['features'].copy()

    if 'abuse_trend_slope' not in df.columns:
        return "Trend data not available."

    # Filter districts with negative slope (decreasing)
    decreasing = df[df['abuse_trend_slope'] < 0].copy()

    if len(decreasing) == 0:
        return "No districts currently show decreasing trends."

    # Sort by improvement (most negative slope first)
    decreasing = decreasing.sort_values('abuse_trend_slope')

    response = "DISTRICTS WITH IMPROVING TRENDS\n"
    response += "="*60 + "\n"
    response += "These districts show successful interventions worth studying\n\n"

    for idx, row in decreasing.iterrows():
        district = row['District']
        slope = row['abuse_trend_slope']
        current_rate = row.get('abuse_rate_per_1000', 0)
        pct_change_3yr = row.get('abuse_pct_change_3yr', 0)

        response += f"""
{district.upper()}
  Trend: {slope:.1f} cases/year (DECREASING)
  Current Rate: {current_rate:.2f} per 1000 children
  3-Year Change: {pct_change_3yr:.1f}%
  STATUS: Positive trend - monitor and maintain programs

"""

    return response

In [ ]:
def compare_districts(districts: List[str]) -> str:
    """
    Compare two or more districts side-by-side.

    WHY: Managers need to understand relative differences
    """
    if len(districts) < 2:
        return "Please specify at least two districts to compare."

    if datasets['features'] is None:
        return "Feature data not available."

    df = datasets['features'].copy()

    comparison_data = []
    for district in districts:
        district_row = df[df['District'].str.lower() == district.lower()]
        if len(district_row) > 0:
            comparison_data.append(district_row.iloc[0])

    if len(comparison_data) == 0:
        return "No data found for specified districts."

    response = "DISTRICT COMPARISON\n"
    response += "="*70 + "\n\n"

    metrics = [
        ('abuse_rate_per_1000', 'Current Abuse Rate (per 1000)', '{:.2f}'),
        ('abuse_trend_slope', 'Trend (cases/year)', '{:+.1f}'),
        ('student_teacher_ratio', 'Student-Teacher Ratio', '{:.1f}'),
        ('schools_per_1000_children', 'Schools per 1000 Children', '{:.2f}'),
        ('total_child_population', 'Total Child Population', '{:,.0f}')
    ]

    for metric_col, metric_name, fmt in metrics:
        if metric_col in df.columns:
            response += f"\n{metric_name}:\n"
            for data in comparison_data:
                district_name = data['District']
                value = data.get(metric_col, 0)
                response += f"  {district_name}: {fmt.format(value)}\n"

    # Add allocation comparison if available
    if datasets['allocations'] is not None:
        response += "\nRecommended Allocation:\n"
        alloc_df = datasets['allocations']
        for district in districts:
            alloc_row = alloc_df[alloc_df['District'].str.lower() == district.lower()]
            if len(alloc_row) > 0:
                pct = alloc_row.iloc[0]['Allocation_Percent']
                response += f"  {district}: {pct:.1f}%\n"

    return response


print("Trend and comparison functions defined")

In [ ]:
# Additional specialized functions

def get_poor_infrastructure() -> str:
    """
    Identify districts with infrastructure problems.

    WHY: Infrastructure investment is part of resource allocation
    """
    if datasets['features'] is None:
        return "Feature data not available."

    df = datasets['features'].copy()

    if 'student_teacher_ratio' not in df.columns:
        return "Infrastructure data not available."

    # Define poor infrastructure as top 75th percentile in student-teacher ratio
    threshold = df['student_teacher_ratio'].quantile(0.75)
    poor_infra = df[df['student_teacher_ratio'] >= threshold].copy()
    poor_infra = poor_infra.sort_values('student_teacher_ratio', ascending=False)

    response = "DISTRICTS WITH INFRASTRUCTURE CHALLENGES\n"
    response += "="*60 + "\n"
    response += f"Threshold: Student-Teacher ratio >= {threshold:.1f}\n\n"

    for idx, row in poor_infra.iterrows():
        district = row['District']
        st_ratio = row['student_teacher_ratio']
        schools_per_1000 = row.get('schools_per_1000_children', 0)

        response += f"""
{district.upper()}
  Student-Teacher Ratio: {st_ratio:.1f}:1 (STRAINED)
  Schools per 1000 children: {schools_per_1000:.2f}
  RECOMMENDATION: Infrastructure investment needed
    - Recruit more teachers
    - Build additional schools/classrooms
    - Improve existing facilities

"""

    return response

In [ ]:
def get_high_current_abuse() -> str:
    """
    Show districts with highest current abuse rates.

    WHY: Immediate crisis response needed
    """
    if datasets['features'] is None:
        return "Feature data not available."

    df = datasets['features'].copy()

    if 'abuse_rate_per_1000' not in df.columns:
        return "Abuse rate data not available."

    # Sort by current abuse rate
    df_sorted = df.sort_values('abuse_rate_per_1000', ascending=False)
    top_3 = df_sorted.head(3)

    response = "DISTRICTS WITH HIGHEST CURRENT ABUSE RATES\n"
    response += "="*60 + "\n"
    response += "These districts require immediate intervention\n\n"

    for idx, row in top_3.iterrows():
        district = row['District']
        rate = row['abuse_rate_per_1000']
        cases = row.get('abuse_cases', 0)
        trend = row.get('abuse_trend_slope', 0)

        trend_text = "INCREASING" if trend > 0 else "DECREASING" if trend < 0 else "STABLE"

        response += f"""
{district.upper()}
  Abuse Rate: {rate:.2f} per 1000 children
  Total Cases: {cases:.0f}
  Trend: {trend_text} ({trend:+.1f} cases/year)
  URGENCY: IMMEDIATE ACTION REQUIRED

"""

    return response

In [ ]:
def get_tier_classification() -> str:
    """
    Show districts organized by priority tier.

    WHY: Different tiers need different intervention strategies
    """
    if datasets['allocations'] is None:
        return "Allocation data not available."

    df = datasets['allocations'].copy()

    response = "DISTRICT CLASSIFICATION BY PRIORITY TIER\n"
    response += "="*70 + "\n\n"

    for tier in sorted(df['Priority_Tier'].unique()):
        tier_districts = df[df['Priority_Tier'] == tier].sort_values('Rank')

        response += f"{tier.upper()}\n"
        response += "-" * 70 + "\n"

        for idx, row in tier_districts.iterrows():
            district = row['District']
            allocation = row['Allocation_Percent']
            response += f"  {district}: {allocation:.1f}% allocation\n"

        response += "\n"

    return response

In [ ]:
def get_district_profile(districts: List[str]) -> str:
    """
    Complete profile for a district.

    WHY: Comprehensive overview for district-specific planning
    """
    if not districts:
        return "Please specify a district."

    district = districts[0]  # Take first district

    # Gather data from all sources
    profile = f"COMPLETE PROFILE: {district.upper()}\n"
    profile += "="*70 + "\n\n"

    # Current status
    if datasets['features'] is not None:
        feat_row = datasets['features'][
            datasets['features']['District'].str.lower() == district.lower()
        ]

        if len(feat_row) > 0:
            row = feat_row.iloc[0]
            profile += "CURRENT STATUS (2024)\n"
            profile += "-"*70 + "\n"
            profile += f"Child Population: {row.get('total_child_population', 0):,.0f}\n"
            profile += f"Abuse Rate: {row.get('abuse_rate_per_1000', 0):.2f} per 1000 children\n"
            profile += f"Total Abuse Cases: {row.get('abuse_cases', 0):.0f}\n"
            profile += f"Student-Teacher Ratio: {row.get('student_teacher_ratio', 0):.1f}:1\n"
            profile += "\n"

            profile += "TRENDS\n"
            profile += "-"*70 + "\n"
            slope = row.get('abuse_trend_slope', 0)
            trend_text = "INCREASING" if slope > 0 else "DECREASING" if slope < 0 else "STABLE"
            profile += f"Abuse Trend: {trend_text} ({slope:+.1f} cases/year)\n"
            profile += f"3-Year Change: {row.get('abuse_pct_change_3yr', 0):+.1f}%\n"
            profile += "\n"

    # Allocation recommendation
    if datasets['allocations'] is not None:
        alloc_row = datasets['allocations'][
            datasets['allocations']['District'].str.lower() == district.lower()
        ]

        if len(alloc_row) > 0:
            row = alloc_row.iloc[0]
            profile += "RESOURCE ALLOCATION\n"
            profile += "-"*70 + "\n"
            profile += f"Priority Rank: {row['Rank']} out of {len(datasets['allocations'])}\n"
            profile += f"Need Score: {row['Need_Score']:.1f}/100\n"
            profile += f"Priority Tier: {row['Priority_Tier']}\n"
            profile += f"Recommended Allocation: {row['Allocation_Percent']:.1f}%\n"
            profile += "\n"

    return profile


print("All specialized response functions defined")

In [ ]:
# Trend analysis and budget scenario functions

def get_trend_analysis(districts: List[str] = None) -> str:
    """
    Show historical trends over time.

    WHY: Understanding past patterns informs future planning
    """
    if datasets['time_series'] is None:
        return "Time series data not available."

    df = datasets['time_series'].copy()

    if districts:
        # Filter to specific districts
        df = df[df['District'].str.lower().isin([d.lower() for d in districts])]

    # Calculate summary statistics
    summary = df.groupby('District').agg({
        'Reported chid abuse cases': ['min', 'max', 'mean']
    }).round(1)

    response = "ABUSE CASE TRENDS (2012-2024)\n"
    response += "="*70 + "\n\n"

    for district in summary.index:
        min_cases = summary.loc[district, ('Reported chid abuse cases', 'min')]
        max_cases = summary.loc[district, ('Reported chid abuse cases', 'max')]
        avg_cases = summary.loc[district, ('Reported chid abuse cases', 'mean')]

        response += f"""
{district.upper()}
  Minimum (2012-2024): {min_cases:.0f} cases
  Maximum (2012-2024): {max_cases:.0f} cases
  Average: {avg_cases:.1f} cases/year
  Range: {max_cases - min_cases:.0f} cases variation

"""

    return response

In [ ]:
def calculate_budget_scenario(budget_amount: float) -> str:
    """
    Calculate allocation for different budget amount.

    WHY: Scenario planning for budget uncertainty
    """
    if datasets['allocations'] is None:
        return "Allocation data not available."

    df = datasets['allocations'].copy()
    df['Scenario_Budget'] = (df['Allocation_Percent'] / 100) * budget_amount
    df['Budget_Formatted'] = df['Scenario_Budget'].apply(format_rupees)

    display_df = df[[
        'Rank', 'District', 'Allocation_Percent', 'Budget_Formatted'
    ]].copy()

    display_df.columns = ['Rank', 'District', 'Allocation %', 'Budget Amount']

    table = display_df.to_string(index=False)

    return f"""
BUDGET SCENARIO ANALYSIS
Scenario Budget: {format_rupees(budget_amount)} Rupees
{'='*80}

{table}

{'='*80}
Total: {format_rupees(budget_amount)} Rupees

NOTE: Allocation percentages remain constant.
      Only rupee amounts change with budget size.
"""


print("Trend and scenario functions defined")

## 8. Main Query Handler

Routes queries to appropriate handler function based on intent.

In [ ]:
def process_query(user_query: str, debug: bool = False) -> str:
    """
    Main query processing pipeline.

    PROCESS:
    1. Classify intent
    2. Extract entities (districts, budget amounts)
    3. Route to appropriate handler
    4. Generate response

    Args:
        user_query: Natural language query from user
        debug: If True, show classification details

    Returns:
        Natural language response
    """
    # Step 1: Classify intent
    intent, confidence, metadata = classify_intent(user_query, debug=debug)

    if debug:
        print(f"\nClassified Intent: {intent}")
        print(f"Confidence: {confidence:.4f}")
        print(f"Matched Template: {metadata['matched_template']}")
        print()

    # Step 2: Extract entities
    districts = extract_district_names(user_query, DISTRICTS)
    budget_amount = extract_budget_amount(user_query)

    if debug and districts:
        print(f"Extracted Districts: {districts}")
    if debug and budget_amount:
        print(f"Extracted Budget: {format_rupees(budget_amount)}")

    # Step 3: Route to handler
    handler_name = metadata['handler']

    try:
        # Call appropriate handler function
        if handler_name == 'get_district_allocation':
            budget = budget_amount if budget_amount else DEFAULT_BUDGET
            return get_district_allocation(districts, budget)

        elif handler_name == 'get_highest_priority':
            return get_highest_priority()

        elif handler_name == 'get_all_allocations':
            budget = budget_amount if budget_amount else DEFAULT_BUDGET
            return get_all_allocations(budget)

        elif handler_name == 'get_increasing_abuse':
            return get_increasing_abuse()

        elif handler_name == 'get_decreasing_abuse':
            return get_decreasing_abuse()

        elif handler_name == 'compare_districts':
            return compare_districts(districts)

        elif handler_name == 'get_poor_infrastructure':
            return get_poor_infrastructure()

        elif handler_name == 'get_high_current_abuse':
            return get_high_current_abuse()

        elif handler_name == 'get_tier_classification':
            return get_tier_classification()

        elif handler_name == 'get_district_profile':
            return get_district_profile(districts)

        elif handler_name == 'get_trend_analysis':
            return get_trend_analysis(districts)

        elif handler_name == 'calculate_budget_scenario':
            if budget_amount:
                return calculate_budget_scenario(budget_amount)
            else:
                return "Please specify a budget amount. Example: 'What if budget is 75 million?'"

        else:
            return f"Handler '{handler_name}' not implemented yet."

    except Exception as e:
        return f"Error processing query: {str(e)}\nPlease try rephrasing your question."


print("Main query processor defined")

In [ ]:
def get_help() -> str:
    """
    Provide help information.
    """
    return """
HELP: SUPPORTED QUERY TYPES
========================================

ALLOCATION QUERIES:
  - "What is the allocation for [district]?"
  - "Show all district allocations"
  - "Which district gets the most?"

TREND QUERIES:
  - "Which districts have increasing abuse?"
  - "Show me improving districts"
  - "Trends in [district]"

COMPARISON QUERIES:
  - "Compare [district1] and [district2]"
  - "How do [district1] and [district2] differ?"

PRIORITY QUERIES:
  - "Which district is highest priority?"
  - "Show tier classifications"
  - "Critical districts"

INFRASTRUCTURE QUERIES:
  - "Districts with poor infrastructure"
  - "Where is infrastructure worst?"

PROFILE QUERIES:
  - "Tell me about [district]"
  - "Complete profile for [district]"

SCENARIO QUERIES:
  - "What if budget is 75 million?"
  - "Calculate allocation for 60 million"

COMMANDS:
  - Type 'exit' to quit
  - Type 'debug on' to see classification details
  - Type 'help' to see this message
"""

print("Additional utility functions defined")

## 9. Interactive Query Interface

User-friendly loop for asking questions.

In [ ]:
# Test the system with sample queries first
print("="*80)
print("TESTING NATURAL LANGUAGE INTERFACE")
print("="*80)
print()

sample_queries = [
    "Which district needs the most resources?",
    "Show me all allocations",
    "Which districts have increasing abuse?"
]

for query in sample_queries:
    print(f"\nQuery: {query}")
    print("-"*80)
    response = process_query(query)
    print(response)
    print()

print("="*80)
print("Test queries completed successfully")
print("="*80)

Loop

In [ ]:
# Interactive query loop
print("\n" + "="*80)
print("NATURAL LANGUAGE INTERFACE FOR CHILD WELFARE DATA")
print("="*80)
print()
print("Welcome! Ask questions about districts, allocations, and trends.")
print()
print("Example questions:")
print("  - Which district needs the most resources?")
print("  - Show me allocation for Hambantota")
print("  - Compare Matara and Colombo")
print("  - Which districts have increasing abuse?")
print("  - Show me districts with poor infrastructure")
print("  - What if budget is 75 million?")
print()
print("Type 'exit' to quit, 'debug on' to see classification details")
print("="*80)

debug_mode = False

while True:
    try:
        print("\n")
        user_query = input("Your question: ").strip()

        if not user_query:
            continue

        # Check for exit
        if user_query.lower() in ['exit', 'quit', 'q']:
            print("\nThank you for using the Natural Language Interface!")
            break

        # Toggle debug mode
        if user_query.lower() == 'debug on':
            debug_mode = True
            print("Debug mode enabled")
            continue

        if user_query.lower() == 'debug off':
            debug_mode = False
            print("Debug mode disabled")
            continue

        # Process query
        print("\n" + "-"*80)
        response = process_query(user_query, debug=debug_mode)
        print(response)
        print("-"*80)

    except KeyboardInterrupt:
        print("\n\nSession interrupted by user.")
        break

    except Exception as e:
        print(f"\nError: {str(e)}")
        print("Please try again with a different question.")

## Summary

This advanced NLP interface provides:

1. **Multi-Dataset Integration**: Queries across time series, features, and allocations
2. **12 Query Intent Categories**: Covers all manager use cases
3. **Fuzzy District Matching**: Handles typos and variations
4. **Budget Amount Extraction**: Understands "75 million", "7.5 crore", etc.
5. **Natural Language Responses**: Human-friendly, actionable insights
6. **Debug Mode**: Shows classification process for validation
7. **Extensible Architecture**: Easy to add new query types

The system transforms complex data into simple conversational answers that welfare managers can use for decision-making.